# Сложные операции над объектами Pandas

На прошлом занятии мы изучили простые операции над объектами Pandas: векторизованные операции, операции для очистки данных, а также операции, которые могут быть полезны для преобразования данных в отдельных столбцах. В этом занятии мы рассмотрим более комплексные операции над объектами Pandas. Эти операции включают в себя объединение датафреймов по индексам и колонкам, а также агрегирование данных.

**Необходимые импорты**:

In [17]:
from collections.abc import Sequence
from typing import Any
from uuid import uuid4

import numpy as np
import pandas as pd

**Вспомогательные объекты**:

In [3]:
def create_df_from_index_and_columns(
    index: Sequence[Any],
    columns: Sequence[Any],
) -> pd.DataFrame:
    data = {
        column_name: [
            f"{index_name}|{column_name}" for index_name in index
        ]
        for column_name in columns
    }

    return pd.DataFrame(data, index=index)

## Объединение данных

Очень часто при решении практических задач вам придется иметь дело не с одним единственным источником данных, а с несколькими. Данные могут храниться в разных csv-файлах, таблицах в БД и даже на разных серверах. Поэтому важно уметь правильно объединять данные из разных источников в единый датафрейм для удобства дальнейшей работы (конечно, если это целесообразно в рамках решаемой задачи). Итак, в этой секции рассмотрим наиболее распространенные способы объединения данных в Pandas.

### Конкатенация

Конкатенация - это простейшая форма объединения двух и более наборов данных в единый набор. Несмотря на свою простоту, конкатенация бывает очень полезна в тех случаях, когда вам нужно объединить данные с одинаковой структурой, полученные из разных источников. Примером таких данных может быть некоторая таблица, которая хранится в виде набора csv-файлов с одинаковой структурой. Чтобы восстановить исходную таблицу из данного набора csv-файлов, нам достаточно просто прочитать содержимое этих файлов и выполнить конкатенацию полученных датафреймов.

Конкатенация в Pandas напоминает конкатенацию в NumPy и выполняется с помощью функции `pd.concat`. Обязательный аргумент функции `pd.concat` - итерируемый объект `objs` с объектами, которые требуется объединить. С помощью этой функции вы можете объединить два и более набора данных (`pd.Series` или `pd.DataFrame`), записав данные из них последовательно, друг за другом.

Простейший вариант использования функции `pd.concat` - конкатенация двух объектов `pd.Series`:

In [4]:
series_size = 3

series1 = pd.Series(
    np.random.randint(170, 190, size=series_size, dtype=np.uint8),
    index=[str(uuid4()) for _ in range(series_size)]
)
series2 = pd.Series(
    np.random.randint(150, 180, size=series_size, dtype=np.uint8),
    index=[str(uuid4()) for _ in range(series_size)]
)
series_concatenated = pd.concat((series1, series2))

print(
    f"series1:\n{series1}",
    f"series2:\n{series2}",
    f"concatenation result:\n{series_concatenated}",
    sep="\n\n",
)

series1:
e24e410c-4e28-41ed-a5bb-1e8d5f69bbb0    189
fd83ecbe-2bf8-44cb-b4c6-f3d86f593d69    178
11053424-b926-4c2b-ba27-02d72df8b67c    177
dtype: uint8

series2:
bd1b07fd-4bfa-4662-b226-c0b0458d68d7    151
bdf25246-c93d-45c8-8b1c-240839e914b0    150
0ddb3bbd-1ce2-45d0-ade3-0242e20e246d    153
dtype: uint8

concatenation result:
e24e410c-4e28-41ed-a5bb-1e8d5f69bbb0    189
fd83ecbe-2bf8-44cb-b4c6-f3d86f593d69    178
11053424-b926-4c2b-ba27-02d72df8b67c    177
bd1b07fd-4bfa-4662-b226-c0b0458d68d7    151
bdf25246-c93d-45c8-8b1c-240839e914b0    150
0ddb3bbd-1ce2-45d0-ade3-0242e20e246d    153
dtype: uint8


Функция `pd.concat` также может быть использована для объединения более сложных объектов, типа `pd.DataFrame`:

In [5]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=[1, 2],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=[3, 4],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
     col1    col2
1  1|col1  1|col2
2  2|col1  2|col2

dataframe2:
     col1    col2
3  3|col1  3|col2
4  4|col1  4|col2

concatenation result:
     col1    col2
1  1|col1  1|col2
2  2|col1  2|col2
3  3|col1  3|col2
4  4|col1  4|col2


По умолчанию объединение объектов `pd.Dataframe` с помощью `pd.concat` выполняется построчно. Однако, при объединении данных высокой размерности `pd.concat` предоставляет возможность настройки, как именно будут объединены данные. За эту настройку отвечает знакомый нам аргумент `axis`. Этот аргумент может быть связан как с целочисленными значениями, как мы уже видели при изучении NumPy, так и со строковыми литералами, типа `"columns"`. Использование строковых литералов предпочтительнее из-за увеличения понятности кода.

In [6]:
index = ["row1", "row2"]

dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=index,
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col3", "col4"],
    index=index,
)
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    axis="columns",
)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col3       col4
row1  row1|col3  row1|col4
row2  row2|col3  row2|col4

concatenation result:
           col1       col2       col3       col4
row1  row1|col1  row1|col2  row1|col3  row1|col4
row2  row2|col1  row2|col2  row2|col3  row2|col4


Во всех предыдущих примерах мы предполагали, что объединяемые объекты имеют или одни и те же индексы, или одни и те же имена столбцов. Однако `pd.concat` будет корректно работать и в тех случаях, когда это не так. Если объединяемые данные имеют разный набор индексов и столбцов, место недостающих данных будет заполнено `NaN` значениями.

In [7]:
dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col3", "col4"],
    index=["row3", "row4"],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col3       col4
row3  row3|col3  row3|col4
row4  row4|col3  row4|col4

concatenation result:
           col1       col2       col3       col4
row1  row1|col1  row1|col2        NaN        NaN
row2  row2|col1  row2|col2        NaN        NaN
row3        NaN        NaN  row3|col3  row3|col4
row4        NaN        NaN  row4|col3  row4|col4


#### Борьба с дублированием индексов

После объединения набора данных функция `pd.concat` сохраняет исходные индексы. Данное поведение может приводить к появлению дублирующихся индексов.

В приведенном примере результирующий `pd.DataFrame` будет содержать два значения `"row2"` в своем индексе.

In [8]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
dataframe_concatenated = pd.concat((dataframe1, dataframe2))

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col1       col2
row2  row2|col1  row2|col2
row3  row3|col1  row3|col2

concatenation result:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2
row2  row2|col1  row2|col2
row3  row3|col1  row3|col2


В некоторых ситуациях такое дублирование нежелательно. `pd.concat` предоставляет два способа по борьбе с дублирующимися индексами: игнорирование индексов исходных наборов данных и проверка уникальности.

При игнорировании индексов исходных наборов данных результат конкатенации будет содержать новый индекс, сгенерированный автоматически. Чтобы воспользоваться данным способом, необходимо вызвать функцию `pd.concat` с флагом `ignore_index=True`.

In [9]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    ignore_index=True,
)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"concatenation result:\n{dataframe_concatenated}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col1       col2
row2  row2|col1  row2|col2
row3  row3|col1  row3|col2

concatenation result:
        col1       col2
0  row1|col1  row1|col2
1  row2|col1  row2|col2
2  row2|col1  row2|col2
3  row3|col1  row3|col2


В этом случае индекс не будет содержать никаких дубликатов, однако при этом вы теряете значения исходных индексов. Поэтому данный способ борьбы с дубликатами подходит в том случае, если значения индекса неважны для решения задачи.

Второй способ борьбы с дубликатами - проверка уникальности. Чтобы воспользоваться данным способом, необходимо вызвать функцию `pd.concat` с флагом `verify_integrity=True`.

In [10]:
columns = ["col1", "col2"]

dataframe1 = create_df_from_index_and_columns(
    columns=columns,
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=columns,
    index=["row2", "row3"],
)
# этот вызов должен привести к ошибке ValueError
dataframe_concatenated = pd.concat(
    (dataframe1, dataframe2),
    verify_integrity=True,
)

ValueError: Indexes have overlapping values: Index(['row2'], dtype='object')

При использовании флага `verify_integrity=True` вызов функции `pd.concat` завершится успехом в том и только в том случае, если индексы исходных наборов данных не имели пересечения. В противном случае вы получите ошибку `ValueError`.

При использовании данного подхода в результате конкатенации сохраняются значения индексов исходных наборов данных.

Существует еще один способ борьбы с дубликатами - создание мульти-индекса. Однако этот способ в нашем курсе рассматриваться не будет (как и мульти-индексы). Если вы заинтересовались темой мульти-индексов, вы можете ознакомиться с ней самостоятельно в [официальной документации](https://pandas.pydata.org/pandas-docs/version/2.2/reference/api/pandas.MultiIndex.html).

### Сложные объединения данных

Сложные объединения данных в Pandas являются реализациями правил реляционной алгебры по объединению данных. Звучит страшно и сложно, но на практике все не так пугающе, как кажется на первый взгляд. Те из вас, кто знаком с SQL, увидят в описанных функция операцию `join`.

Сложные объединения данных незаменимы в тех случаях, когда данные, полученные из разных источников (например, из разных файлов), имеют сложные системы связей. Например, у нас может быть датасет с данными о пользователях и отдельный датасет с данными об активности пользователей на нашем сайте. Сами по себе датасеты имеют разную структуру и связаны между собой через значения колонки `user_id`. В этом случае для комплексного анализа данных об активности пользователей нашего сайта нам потребуется сложное объединение этих данных, а не простая конкатенация.

В Pandas сложное объединение наборов данных может быть выполнено с помощью функции `pd.merge`.

Чтобы продемонстрировать возможности функции `pd.merge`, рассмотрим пример. Предположим мы занимаемся анализом данных о покупках пользователей в нашем онлайн магазине. Данные хранятся в отдельных csv-файлах:
- [user](./data/user.csv) - данные о пользователях. В этом файле хранятся уникальные ID пользователей, а также персональные данные: username и email.
- [item](./data/item.csv) - данные о товарах. В этом файле хранятся ID товаров, их наименования и стоимости.
- [order_to_user](./data/order_to_user.csv) - связи между пользователями и ID заказов, которые эти пользователи совершали.
- [order_to_item](./data/order_to_item.csv) - связи между заказами и ID позиций, которые входят в заказ.

Загрузим эти данные и посмотрим на них:

In [42]:
user_data = pd.read_csv("./data/user.csv")
item_data = pd.read_csv("./data/item.csv")
order_to_user = pd.read_csv("./data/order_to_user.csv")
order_to_item = pd.read_csv("./data/order_to_item.csv")

print("user data:")
display(user_data)

print("item data:")
display(item_data)

user data:


,user_id,username,email
0,b038434e-c697-4f47-a56c-12ef56c61363,stevenrangel,stevenrangel@example.org
1,43040196-37c5-48e6-9782-9266e63ecd68,wsmith,wsmith@example.net
2,d1dee9a9-6268-4b67-bc27-a6c0641324f4,nlane,nlane@example.com
3,30f0dff3-8ba3-40c4-a62f-821c0ed09784,turnerrodney,turnerrodney@example.net


item data:


,item_id,name,price
0,b63a29e9-dda8-40d6-83b0-dc31c0bdda09,Big Muff PI,$109
1,179b4de0-0d7a-455b-8b9a-fc3e738786b4,Boss OC-5,$120
2,b4763893-fd95-494c-ba7e-1d1ccef1fcbe,Boss RC-1,$109.99
3,8dd3c660-e671-4f07-8af4-4fac9dc5a67f,Fender Rumble 40,$269.99
4,c68f85ba-79dd-4af4-b62d-3b797b998e1c,Orange Crush Bass 50,$399
5,7a98a5b7-6cc3-4a3a-a83e-c13f85fa04ec,Electric Guitar Strings,$8.99
6,c9f9c750-a11c-4186-8172-7b3bda8f1322,Musical Cable 10ft,$27.99
7,fe8d7935-6a74-454b-8417-efd2f463f960,Squier Affinity Jazz Bass,$370


Итак, чтобы получить полные данные, которые бы включали в себя информацию о том, какой покупатель приобрел какую позицию в нашем магазине, нам необходимо объединить разрозненные данные. Очевидно, что все датафреймы имеют разную структуру, поэтому простая конкатенация нам не поможет. Однако датафреймы связаны между собой следующим образом:
- Датафрейм `user_data` связана с датафреймом `order_to_user` через столбец `user_id`.
- Датафрейм `item_data` связана с датафреймом `order_to_item` через столбец `item_id`.
- Датафрейм `order_to_user` связана с датафреймом `order_to_item` через столбец `order_id`.

Используя эти связи, мы можем объединить данные в один датафрейм с помощью функции `pd.merge`:

In [43]:
order_to_item_expended = pd.merge(order_to_item, item_data)
order_to_user_extended = pd.merge(order_to_user, user_data)

pd.merge(order_to_user_extended, order_to_item_expended)

,user_id,order_id,username,email,item_id,name,price
0,b038434e-c697-4f47-a56c-12ef56c61363,325b7ae1-b4bd-42bb-8a08-5771c37293de,stevenrangel,stevenrangel@example.org,b63a29e9-dda8-40d6-83b0-dc31c0bdda09,Big Muff PI,$109
1,b038434e-c697-4f47-a56c-12ef56c61363,325b7ae1-b4bd-42bb-8a08-5771c37293de,stevenrangel,stevenrangel@example.org,7a98a5b7-6cc3-4a3a-a83e-c13f85fa04ec,Electric Guitar Strings,$8.99
2,43040196-37c5-48e6-9782-9266e63ecd68,c1d0dfa3-1e11-4044-9476-44f97677baf1,wsmith,wsmith@example.net,fe8d7935-6a74-454b-8417-efd2f463f960,Squier Affinity Jazz Bass,$370
3,43040196-37c5-48e6-9782-9266e63ecd68,c1d0dfa3-1e11-4044-9476-44f97677baf1,wsmith,wsmith@example.net,c9f9c750-a11c-4186-8172-7b3bda8f1322,Musical Cable 10ft,$27.99
4,43040196-37c5-48e6-9782-9266e63ecd68,c1d0dfa3-1e11-4044-9476-44f97677baf1,wsmith,wsmith@example.net,8dd3c660-e671-4f07-8af4-4fac9dc5a67f,Fender Rumble 40,$269.99
5,43040196-37c5-48e6-9782-9266e63ecd68,c1d0dfa3-1e11-4044-9476-44f97677baf1,wsmith,wsmith@example.net,179b4de0-0d7a-455b-8b9a-fc3e738786b4,Boss OC-5,$120
6,d1dee9a9-6268-4b67-bc27-a6c0641324f4,cd96ef2d-d004-4003-bbfa-abb08f720cf6,nlane,nlane@example.com,fe8d7935-6a74-454b-8417-efd2f463f960,Squier Affinity Jazz Bass,$370
7,b038434e-c697-4f47-a56c-12ef56c61363,f52ad15d-61f1-4fab-9c53-773cf46574da,stevenrangel,stevenrangel@example.org,b4763893-fd95-494c-ba7e-1d1ccef1fcbe,Boss RC-1,$109.99
8,d1dee9a9-6268-4b67-bc27-a6c0641324f4,1154efb1-2d4a-40ad-9147-f450f15d9aed,nlane,nlane@example.com,c68f85ba-79dd-4af4-b62d-3b797b998e1c,Orange Crush Bass 50,$399
9,30f0dff3-8ba3-40c4-a62f-821c0ed09784,6ff03d6f-7a5e-404d-b501-402b3f147023,turnerrodney,turnerrodney@example.net,b63a29e9-dda8-40d6-83b0-dc31c0bdda09,Big Muff PI,$109


За один раз функция `pd.merge` может объединить только два `pd.DataFrame`, поэтому в примере выше нам пришлось вызвать ее три раза. По умолчанию эта функция выполняет поиск столбцов с одинаковыми именами и использует значения этих столбцов в качестве ключей слияния. Проще говоря по умолчанию `pd.merge` делает следующее:
- Находит столбцы с одинаковыми именами в правом и в левом `pd.DataFrame`.
- Для каждой строки левого `pd.DataFrame` ищет строку в правом `pd.DataFrame` такую, что в столбцах, найденных на предыдущем шаге, находятся одинаковые значения.
- Если такая строка найдена, то строка из левого `pd.DataFrame` объединяется со строкой из правого `pd.DataFrame`. Результирующая строка сохраняется в результирующий `pd.DataFrame`.

Однако, к сожалению, на практике далеко не всегда удается использовать дефолтное поведение `pd.merge`. В первую очередь это связано с неоднородностью данных: столбцы с одинаковыми данными в разных `pd.DataFrame` могут иметь разные названия.

In [44]:
order_to_user = order_to_user.rename(columns={"user_id": "UserID"})
# этот вызов приведет к ошибке MergeError,
# т.к. датафреймы не содержат общих колонок
pd.merge(user_data, order_to_user)

MergeError: No common columns to perform merge on. Merge options: left_on=None, right_on=None, left_index=False, right_index=False

В данном примере датафрейм `user_data` содержит столбец `user_id`, а датафрейм `order_to_user` - `UserID`. Поскольку датафреймы не имеют общих столбцов, использовать поведение `pd.merge` по умолчанию невозможно.

На этот случай `pd.merge` позволяет явно указать, какой столбец левого датафрейма и какой столбец правого датафрейма стоит использовать в качестве ключей слияния. Чтобы это сделать, необходимо связать имена столбцов слияния левого и правого датафреймов с аргументами `left_on` и `right_on`, соответственно:

In [45]:
pd.merge(
    user_data,
    order_to_user,
    left_on="user_id",
    right_on="UserID",
)

,user_id,username,email,UserID,order_id
0,b038434e-c697-4f47-a56c-12ef56c61363,stevenrangel,stevenrangel@example.org,b038434e-c697-4f47-a56c-12ef56c61363,325b7ae1-b4bd-42bb-8a08-5771c37293de
1,b038434e-c697-4f47-a56c-12ef56c61363,stevenrangel,stevenrangel@example.org,b038434e-c697-4f47-a56c-12ef56c61363,f52ad15d-61f1-4fab-9c53-773cf46574da
2,b038434e-c697-4f47-a56c-12ef56c61363,stevenrangel,stevenrangel@example.org,b038434e-c697-4f47-a56c-12ef56c61363,1b75832c-4fd3-40f9-907e-cee0784bf850
3,43040196-37c5-48e6-9782-9266e63ecd68,wsmith,wsmith@example.net,43040196-37c5-48e6-9782-9266e63ecd68,c1d0dfa3-1e11-4044-9476-44f97677baf1
4,d1dee9a9-6268-4b67-bc27-a6c0641324f4,nlane,nlane@example.com,d1dee9a9-6268-4b67-bc27-a6c0641324f4,cd96ef2d-d004-4003-bbfa-abb08f720cf6
5,d1dee9a9-6268-4b67-bc27-a6c0641324f4,nlane,nlane@example.com,d1dee9a9-6268-4b67-bc27-a6c0641324f4,1154efb1-2d4a-40ad-9147-f450f15d9aed
6,30f0dff3-8ba3-40c4-a62f-821c0ed09784,turnerrodney,turnerrodney@example.net,30f0dff3-8ba3-40c4-a62f-821c0ed09784,6ff03d6f-7a5e-404d-b501-402b3f147023
7,30f0dff3-8ba3-40c4-a62f-821c0ed09784,turnerrodney,turnerrodney@example.net,30f0dff3-8ba3-40c4-a62f-821c0ed09784,8a980646-4238-4425-b52c-b7c0363afd4d


В результирующий датафрейм будут включены оба столбца. Если вас не устраивает подобное поведение, вы можете явно избавиться от нежелательного столбца с помощью метода `pd.DataFrame.drop`.

В тех случаях, когда датафреймы содержат информативный индекс, `pd.merge` позволяет выполнить слияние по индексам. Для этого достаточно передать флаг `left_index=True`, если в качестве ключа слияния для левого датафрейма следует использовать его индекс, и `right_index=True`, если в качестве ключа слияния для правого датафрейма нужно использовать его индекс.

In [46]:
pd.merge(
    item_data,
    order_to_item,
    left_index=True,
    right_index=True,
)

,item_id_x,name,price,order_id,item_id_y
0,b63a29e9-dda8-40d6-83b0-dc31c0bdda09,Big Muff PI,$109,325b7ae1-b4bd-42bb-8a08-5771c37293de,b63a29e9-dda8-40d6-83b0-dc31c0bdda09
1,179b4de0-0d7a-455b-8b9a-fc3e738786b4,Boss OC-5,$120,325b7ae1-b4bd-42bb-8a08-5771c37293de,7a98a5b7-6cc3-4a3a-a83e-c13f85fa04ec
2,b4763893-fd95-494c-ba7e-1d1ccef1fcbe,Boss RC-1,$109.99,cd96ef2d-d004-4003-bbfa-abb08f720cf6,fe8d7935-6a74-454b-8417-efd2f463f960
3,8dd3c660-e671-4f07-8af4-4fac9dc5a67f,Fender Rumble 40,$269.99,c1d0dfa3-1e11-4044-9476-44f97677baf1,fe8d7935-6a74-454b-8417-efd2f463f960
4,c68f85ba-79dd-4af4-b62d-3b797b998e1c,Orange Crush Bass 50,$399,c1d0dfa3-1e11-4044-9476-44f97677baf1,c9f9c750-a11c-4186-8172-7b3bda8f1322
5,7a98a5b7-6cc3-4a3a-a83e-c13f85fa04ec,Electric Guitar Strings,$8.99,c1d0dfa3-1e11-4044-9476-44f97677baf1,8dd3c660-e671-4f07-8af4-4fac9dc5a67f
6,c9f9c750-a11c-4186-8172-7b3bda8f1322,Musical Cable 10ft,$27.99,c1d0dfa3-1e11-4044-9476-44f97677baf1,179b4de0-0d7a-455b-8b9a-fc3e738786b4
7,fe8d7935-6a74-454b-8417-efd2f463f960,Squier Affinity Jazz Bass,$370,f52ad15d-61f1-4fab-9c53-773cf46574da,b4763893-fd95-494c-ba7e-1d1ccef1fcbe


Для упрощения слияния по индексам в объектах `pd.DataFrame` и `pd.Series` реализован метод `join`.

При необходимости вы можете комбинировать ключи слияния: использовать в качестве ключа слияния для одного датафрейма индекс, а для второго - столбец:

In [47]:
user_data.set_index("user_id", inplace=True)
pd.merge(
    user_data,
    order_to_user,
    left_index=True,
    right_on="UserID",
)

,username,email,UserID,order_id
0,stevenrangel,stevenrangel@example.org,b038434e-c697-4f47-a56c-12ef56c61363,325b7ae1-b4bd-42bb-8a08-5771c37293de
3,stevenrangel,stevenrangel@example.org,b038434e-c697-4f47-a56c-12ef56c61363,f52ad15d-61f1-4fab-9c53-773cf46574da
6,stevenrangel,stevenrangel@example.org,b038434e-c697-4f47-a56c-12ef56c61363,1b75832c-4fd3-40f9-907e-cee0784bf850
1,wsmith,wsmith@example.net,43040196-37c5-48e6-9782-9266e63ecd68,c1d0dfa3-1e11-4044-9476-44f97677baf1
2,nlane,nlane@example.com,d1dee9a9-6268-4b67-bc27-a6c0641324f4,cd96ef2d-d004-4003-bbfa-abb08f720cf6
4,nlane,nlane@example.com,d1dee9a9-6268-4b67-bc27-a6c0641324f4,1154efb1-2d4a-40ad-9147-f450f15d9aed
5,turnerrodney,turnerrodney@example.net,30f0dff3-8ba3-40c4-a62f-821c0ed09784,6ff03d6f-7a5e-404d-b501-402b3f147023
7,turnerrodney,turnerrodney@example.net,30f0dff3-8ba3-40c4-a62f-821c0ed09784,8a980646-4238-4425-b52c-b7c0363afd4d


По умолчанию при объединении данных с помощью `pd.merge` результирующий датафрейм будет содержать только те строки, для которых совпали значения ключей слияния. Если для какой-то строки одного из операндов не было найдено строки с соответствующим значением ключа слияния в другом операнде, такая строка включена в результат не будет.

In [52]:
dataframe1 = create_df_from_index_and_columns(
    columns=["col1", "col2"],
    index=["row1", "row2"],
)
dataframe2 = create_df_from_index_and_columns(
    columns=["col2", "col3"],
    index=["row2", "row3"]
)
dataframe_merged = pd.merge(dataframe1, dataframe2)

print(
    f"dataframe1:\n{dataframe1}",
    f"dataframe2:\n{dataframe2}",
    f"dataframe merged:\n{dataframe_merged}",
    sep="\n\n",
)

dataframe1:
           col1       col2
row1  row1|col1  row1|col2
row2  row2|col1  row2|col2

dataframe2:
           col2       col3
row2  row2|col2  row2|col3
row3  row3|col2  row3|col3

dataframe merged:
        col1       col2       col3
0  row2|col1  row2|col2  row2|col3


С одной стороны, это логичное поведение. С другой же стороны, на практике это поведение не всегда может быть удобным. Для выбора стратегии объединения данных необходимо использовать параметр `how` функции `pd.merge`. По умолчанию значение этого параметра `how="inner"`.

Помимо уже рассмотренного внутреннего объединения, существую следующие стратегии объединения:
- `outer` - в результат включаются все строки из обоих операндов. Отсутствующие данные заполняются значениями `NaN`:

In [53]:
pd.merge(dataframe1, dataframe2, how="outer")

,col1,col2,col3
0,row1|col1,row1|col2,NaN
1,row2|col1,row2|col2,row2|col3
2,NaN,row3|col2,row3|col3


- `left` - в результат включаются все строки из левого операнда. Отсутствующие данные заполняются значениями `NaN`:

In [54]:
pd.merge(dataframe1, dataframe2, how="left")

,col1,col2,col3
0,row1|col1,row1|col2,NaN
1,row2|col1,row2|col2,row2|col3


- `right` - в результат включаются все строки из правого операнда. Отсутствующие данные заполняются значениями `NaN`:

In [55]:
pd.merge(dataframe1, dataframe2, how="right")

,col1,col2,col3
0,row2|col1,row2|col2,row2|col3
1,NaN,row3|col2,row3|col3
